# CE 497 — Sea-Ice Segmentation with Segment Anything Model (SAM)

This notebook uses Meta's **Segment Anything Model (SAM)** to automatically segment sea-ice imagery. For reproducibility, the SAM source and ViT-B checkpoint are archived in **CE497_AI-Civil**.

We compare:
- **Trial 1:** segment the full image once.
- **Trial 2:** split the image into an \(N\times N\) grid, segment each tile, and map the masks back to full-image coordinates.

Tiling makes small floes larger relative to SAM's model input, which can improve small-object detection.

> No manual point, box, or text prompts are used; both trials use `SamAutomaticMaskGenerator`.

**Colab:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.


## 1. Install the CE497 archived copy of Segment Anything

Install the frozen `sam-v1` source from `CE497_AI-Civil/third_party/segment-anything`.


In [ ]:
# Install the archived SAM source and required plotting/image packages.
!pip -q install "git+https://github.com/olivmeng/CE497_AI-Civil.git@sam-v1#subdirectory=third_party/segment-anything"
!pip -q install opencv-python matplotlib


## 2. Import packages and check the GPU


In [ ]:
import os, time, urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: SAM will be much slower on CPU.")
    print("In Colab, select Runtime → Change runtime type → T4 GPU.")


## 3. Download the sea-ice image from the CE 497 GitHub repository

The notebook downloads `CE497_AI-Civil/Figures/FrontierPaper_sea ice.png` automatically.


In [ ]:
IMAGE_URL = ("https://raw.githubusercontent.com/olivmeng/CE497_AI-Civil/main/"
             "Figures/FrontierPaper_sea%20ice.png")
IMAGE_PATH = "/content/sea_ice.png"
urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)
print("Downloaded image to:", IMAGE_PATH)


## 4. Load and display the original image


In [ ]:
image_bgr = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise RuntimeError("The image could not be loaded.")

image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
H, W = image.shape[:2]

# Physical scale: change only IMAGE_WIDTH_M for another image.
IMAGE_WIDTH_M = 1500.0
meters_per_pixel = IMAGE_WIDTH_M / W
pixel_area_m2 = meters_per_pixel**2
IMAGE_HEIGHT_M = H * meters_per_pixel

print(f"Image size:       width={W}, height={H} pixels")
print(f"Physical width:   {IMAGE_WIDTH_M:.2f} m")
print(f"Physical height:  {IMAGE_HEIGHT_M:.2f} m")
print(f"Image resolution: {meters_per_pixel:.6f} m/pixel")
print(f"Pixel area:       {pixel_area_m2:.6f} m^2/pixel")

plt.figure(figsize=(10, 10))
plt.imshow(image)
plt.title(f"Original Sea-Ice Image — {IMAGE_WIDTH_M:.0f} m wide")
plt.axis("off")
plt.show()


## 5. Download the archived pretrained SAM checkpoint

Use the archived **ViT-B** checkpoint (`sam_vit_b_01ec64.pth`) from the CE497 `sam-v1` release and verify its SHA-256 hash.


In [ ]:
import hashlib

SAM_CHECKPOINT_URL = ("https://github.com/olivmeng/CE497_AI-Civil/"
                      "releases/download/sam-v1/sam_vit_b_01ec64.pth")
SAM_CHECKPOINT = "/content/sam_vit_b_01ec64.pth"
EXPECTED_SHA256 = "ec2df62732614e57411cdcf32a23ffdf28910380d03139ee0f4fcbe91eb8c912"

if not os.path.exists(SAM_CHECKPOINT):
    print("Downloading archived SAM ViT-B checkpoint from CE497_AI-Civil...")
    urllib.request.urlretrieve(SAM_CHECKPOINT_URL, SAM_CHECKPOINT)

sha256 = hashlib.sha256()
with open(SAM_CHECKPOINT, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)
actual_sha256 = sha256.hexdigest()

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(f"SAM checkpoint SHA-256 mismatch. Expected {EXPECTED_SHA256}, got {actual_sha256}.")

print("SAM checkpoint:", SAM_CHECKPOINT)
print("SHA-256 verified:", actual_sha256)


## 6. Load SAM


In [ ]:
MODEL_TYPE = "vit_b"
sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)
sam.eval()
print("SAM loaded successfully.")


## 7. Helper functions


In [ ]:
def make_mask_generator(model):
    """Create SAM's automatic mask generator."""
    return SamAutomaticMaskGenerator(
        model=model, points_per_side=48, pred_iou_thresh=0.85,
        stability_score_thresh=0.90, crop_n_layers=1,
        crop_n_points_downscale_factor=2, min_mask_region_area=30,
    )

def create_mask_overlay(image_rgb, masks, alpha=0.62, random_seed=42):
    """Overlay SAM masks; draw large masks first."""
    rng = np.random.default_rng(random_seed)
    result = image_rgb.astype(np.float32).copy()
    for m in sorted(masks, key=lambda x: x["area"], reverse=True):
        mask, color = m["segmentation"], rng.integers(0, 256, size=3)
        result[mask] = (1 - alpha) * result[mask] + alpha * color
    return np.clip(result, 0, 255).astype(np.uint8)

def masks_to_label_image(masks, image_shape):
    """Convert SAM masks to integer labels (0 = background)."""
    labels = np.zeros(image_shape[:2], dtype=np.uint16)
    for object_id, m in enumerate(sorted(masks, key=lambda x: x["area"], reverse=True), 1):
        labels[m["segmentation"]] = object_id
    return labels

def show_three_panel(original, overlay, labels, title):
    """Show original, colored masks, and labels."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax, img, ttl, cmap in [
        (axes[0], original, "Original", None),
        (axes[1], overlay, title, None),
        (axes[2], labels, "Object-label mask", "nipy_spectral"),
    ]:
        ax.imshow(img, cmap=cmap)
        ax.set_title(ttl)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# Trial 1 — Automatic SAM segmentation of the entire image

SAM receives the **entire image** and automatically proposes masks with no manual prompt.


## 8. Run SAM on the full image


In [ ]:
mask_generator = make_mask_generator(sam)
start_time = time.time()
with torch.inference_mode():
    masks_full = mask_generator.generate(image)
time_full = time.time() - start_time

print("Trial 1 complete.")
print("Number of masks detected:", len(masks_full))
print(f"Runtime: {time_full:.1f} s")


## 9. Visualize Trial 1


In [ ]:
# Filter dark water/background masks.
gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
otsu_threshold, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
MIN_ICE_FRACTION = 0.80
MAX_FLOE_AREA_FRACTION = 0.20
print(f"Otsu ice/water brightness threshold = {otsu_threshold:.1f}")

masks_full_floes, masks_rejected = [], []
for m in masks_full:
    seg = m["segmentation"].astype(bool)
    ice_fraction = np.mean(gray[seg] >= otsu_threshold)
    is_floe = (ice_fraction >= MIN_ICE_FRACTION and
               m["area"] / (H * W) <= MAX_FLOE_AREA_FRACTION)
    if is_floe:
        m_keep = m.copy()
        m_keep["ice_fraction"] = ice_fraction
        masks_full_floes.append(m_keep)
    else:
        masks_rejected.append(m)

print(f"\nMASK FILTERING\n==============")
print(f"Original SAM masks:        {len(masks_full)}")
print(f"Retained floe masks:       {len(masks_full_floes)}")
print(f"Rejected background masks: {len(masks_rejected)}")

overlay_full = create_mask_overlay(image, masks_full_floes, alpha=0.62, random_seed=42)
labels_full = masks_to_label_image(masks_full_floes, image.shape)

# Exclude floes touching the outer image boundary from the FSD.
masks_full_fsd = []
for m in masks_full_floes:
    seg = m["segmentation"]
    touches_edge = seg[0, :].any() or seg[-1, :].any() or seg[:, 0].any() or seg[:, -1].any()
    if not touches_edge:
        masks_full_fsd.append(m)

print(f"\nFloes used for FSD:      {len(masks_full_fsd)}")
print(f"Boundary floes excluded: {len(masks_full_floes) - len(masks_full_fsd)}")

areas_full_pixels = np.array([m["area"] for m in masks_full_fsd], dtype=float)
areas_full_m2 = areas_full_pixels * meters_per_pixel**2
diameters_full_m = 2 * np.sqrt(areas_full_m2 / np.pi)

# Original | floe overlay | labels | FSD
fig, axes = plt.subplots(1, 4, figsize=(26, 7))
panels = [
    (image, "Original image", None),
    (overlay_full, f"Sea-ice floes only\n{len(masks_full_floes)} masks", None),
    (labels_full, "Floe-label mask", "nipy_spectral"),
]
for ax, (img, title, cmap) in zip(axes[:3], panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

if len(diameters_full_m) >= 2:
    bin_edges = np.logspace(np.log10(diameters_full_m.min()),
                            np.log10(diameters_full_m.max()), 30)
    pdf, edges = np.histogram(diameters_full_m, bins=bin_edges, density=True)
    centers = np.sqrt(edges[:-1] * edges[1:])
    valid = pdf > 0
    axes[3].plot(centers[valid], pdf[valid], marker="o")
    axes[3].set(xscale="log", yscale="log",
                xlabel="Equivalent floe diameter $D_f$ [m]",
                ylabel="Probability density $p(D_f)$ [1/m]",
                title="Floe-size distribution")
    axes[3].grid(True, which="both", alpha=0.3)
else:
    axes[3].text(0.5, 0.5, "Not enough floes\nfor FSD",
                 ha="center", va="center", transform=axes[3].transAxes)
    axes[3].set_title("Floe-size distribution")

plt.suptitle(f"Trial 1 — Full-image SAM | image width = {IMAGE_WIDTH_M:.0f} m", fontsize=16)
plt.tight_layout()
plt.show()

if len(diameters_full_m):
    print("\nFLOE-SIZE STATISTICS\n====================")
    print(f"Minimum diameter: {diameters_full_m.min():.2f} m")
    print(f"Median diameter:  {np.median(diameters_full_m):.2f} m")
    print(f"Mean diameter:    {np.mean(diameters_full_m):.2f} m")
    print(f"Maximum diameter: {diameters_full_m.max():.2f} m")


### What does one SAM mask contain?

- `segmentation` — binary mask
- `area` — area in pixels
- `bbox` — bounding box
- `predicted_iou` — estimated mask quality
- `stability_score` — mask stability


In [ ]:
if masks_full:
    example = masks_full[0]
    print("Keys:", example.keys())
    print("\nArea:", example["area"], "pixels")
    print("Bounding box [x, y, width, height]:", example["bbox"])
    print("Predicted IoU:", example["predicted_iou"])
    print("Stability score:", example["stability_score"])


# Trial 2 — Divide the image into an \(N \times N\) grid

SAM runs independently on each tile, then each mask is mapped back to full-image coordinates.

**Why tiling can help:** SAM resizes its input, so small floes can become difficult to resolve in the full image. Tiling makes each floe occupy a larger fraction of the model input.

**Limitation:** floes crossing tile boundaries can be split into multiple masks.


## 10. Choose \(N\)

Try `N = 2` (4 tiles), `N = 3` (9 tiles), or `N = 4` (16 tiles). Start with **N = 3**.


In [ ]:
N = 3
assert isinstance(N, int) and N >= 1
print(f"Dividing the image into {N} × {N} = {N*N} tiles.")


## 11. Divide each image dimension into N equal parts

`numpy.linspace` defines tile boundaries so every image pixel is included exactly once.


In [ ]:
x_edges = np.linspace(0, W, N + 1, dtype=int)
y_edges = np.linspace(0, H, N + 1, dtype=int)
print("x boundaries:", x_edges)
print("y boundaries:", y_edges)

plt.figure(figsize=(10, 10))
plt.imshow(image)
for x in x_edges[1:-1]:
    plt.axvline(x, linewidth=2)
for y in y_edges[1:-1]:
    plt.axhline(y, linewidth=2)
for row in range(N):
    for col in range(N):
        x0, x1 = x_edges[col:col+2]
        y0, y1 = y_edges[row:row+2]
        plt.text((x0+x1)/2, (y0+y1)/2, f"({row},{col})",
                 ha="center", va="center", fontsize=12,
                 bbox=dict(facecolor="white", alpha=0.7))
plt.title(f"Trial 2 — Image divided into {N} × {N} tiles")
plt.xlim(0, W)
plt.ylim(H, 0)
plt.axis("off")
plt.show()


## 12. Preview the image tiles


In [ ]:
fig, axes = plt.subplots(N, N, figsize=(14, 14))
axes = np.atleast_2d(axes)

for row in range(N):
    for col in range(N):
        x0, x1 = x_edges[col:col+2]
        y0, y1 = y_edges[row:row+2]
        tile = image[y0:y1, x0:x1]
        axes[row, col].imshow(tile)
        axes[row, col].set_title(f"Tile ({row},{col})\n{tile.shape[1]} × {tile.shape[0]} px")
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()


## 13. Run automatic SAM segmentation independently on every tile

For each tile: extract it, run automatic SAM, map masks back to full-image coordinates, and record the source tile. No manual prompt is used.


In [ ]:
def segment_image_by_tiles(image_rgb, model, N):
    """Run automatic SAM on N×N tiles and map masks to full-image coordinates."""
    height, width = image_rgb.shape[:2]
    x_edges_local = np.linspace(0, width, N + 1, dtype=int)
    y_edges_local = np.linspace(0, height, N + 1, dtype=int)
    generator = make_mask_generator(model)
    all_global_masks = []
    tile_counts = np.zeros((N, N), dtype=int)

    for tile_number, (row, col) in enumerate(
        ((r, c) for r in range(N) for c in range(N)), start=1
    ):
        x0, x1 = x_edges_local[col:col+2]
        y0, y1 = y_edges_local[row:row+2]
        tile = image_rgb[y0:y1, x0:x1]
        print(f"Tile {tile_number:02d}/{N*N}: row={row}, col={col}, "
              f"x=[{x0}:{x1}], y=[{y0}:{y1}], size={tile.shape[1]}×{tile.shape[0]}")

        with torch.inference_mode():
            local_masks = generator.generate(tile)
        tile_counts[row, col] = len(local_masks)

        for local in local_masks:
            global_seg = np.zeros((height, width), dtype=bool)
            global_seg[y0:y1, x0:x1] = local["segmentation"]
            global_mask = dict(local)
            global_mask["segmentation"] = global_seg
            bx, by, bw, bh = local["bbox"]
            global_mask["bbox"] = [bx + x0, by + y0, bw, bh]
            global_mask["tile_row"], global_mask["tile_col"] = row, col
            all_global_masks.append(global_mask)

        if device == "cuda":
            torch.cuda.empty_cache()

    return all_global_masks, tile_counts

start_time = time.time()
masks_tiled, tile_counts = segment_image_by_tiles(image, sam, N=N)
time_tiled = time.time() - start_time
print("\nTrial 2 complete.")
print("Total number of tiled masks:", len(masks_tiled))
print(f"Runtime: {time_tiled:.1f} s")


## 14. Number of masks detected in each tile


In [ ]:
print("Mask counts by tile:")
print(tile_counts)

plt.figure(figsize=(7, 6))
plt.imshow(tile_counts)
for row in range(N):
    for col in range(N):
        plt.text(col, row, str(tile_counts[row, col]), ha="center", va="center", fontsize=13)
plt.title("Number of SAM masks detected in each tile")
plt.xlabel("Tile column")
plt.ylabel("Tile row")
plt.xticks(range(N))
plt.yticks(range(N))
plt.colorbar(label="Number of masks")
plt.show()


## 15. Stitch the tile masks back together and visualize Trial 2


In [ ]:
# Apply the same brightness filter to tiled masks.
masks_tiled_floes, masks_tiled_rejected = [], []
for m in masks_tiled:
    seg = m["segmentation"].astype(bool)
    ice_fraction = np.mean(gray[seg] >= otsu_threshold)
    if ice_fraction >= MIN_ICE_FRACTION:
        m_keep = m.copy()
        m_keep["ice_fraction"] = ice_fraction
        masks_tiled_floes.append(m_keep)
    else:
        masks_tiled_rejected.append(m)

print("MASK FILTERING — TRIAL 2\n========================")
print(f"Original tiled SAM masks: {len(masks_tiled)}")
print(f"Retained floe masks:      {len(masks_tiled_floes)}")
print(f"Rejected masks:           {len(masks_tiled_rejected)}")

overlay_tiled = create_mask_overlay(image, masks_tiled_floes, alpha=0.62, random_seed=42)
labels_tiled = masks_to_label_image(masks_tiled_floes, image.shape)

# Exclude outer-image and internal-tile boundary floes from the FSD.
x_edges_fsd = np.linspace(0, W, N + 1, dtype=int)
y_edges_fsd = np.linspace(0, H, N + 1, dtype=int)
internal_x_edges, internal_y_edges = x_edges_fsd[1:-1], y_edges_fsd[1:-1]

masks_tiled_fsd = []
n_outer_boundary = n_tile_boundary = 0
for m in masks_tiled_floes:
    seg = m["segmentation"].astype(bool)
    touches_outer = seg[0, :].any() or seg[-1, :].any() or seg[:, 0].any() or seg[:, -1].any()
    touches_internal = (
        any(seg[:, x].any() or seg[:, x-1].any() for x in internal_x_edges) or
        any(seg[y, :].any() or seg[y-1, :].any() for y in internal_y_edges)
    )
    if touches_outer:
        n_outer_boundary += 1
    elif touches_internal:
        n_tile_boundary += 1
    else:
        masks_tiled_fsd.append(m)

print("\nFSD QUALITY CONTROL\n===================")
print(f"Retained floe masks:          {len(masks_tiled_floes)}")
print(f"Outer-boundary floes removed: {n_outer_boundary}")
print(f"Tile-boundary floes removed:  {n_tile_boundary}")
print(f"Floes used for FSD:           {len(masks_tiled_fsd)}")

areas_tiled_pixels = np.array([m["area"] for m in masks_tiled_fsd], dtype=float)
areas_tiled_m2 = areas_tiled_pixels * meters_per_pixel**2
diameters_tiled_m = 2 * np.sqrt(areas_tiled_m2 / np.pi)

# Original | floe overlay | labels | FSD
fig, axes = plt.subplots(1, 4, figsize=(26, 7))
panels = [
    (image, "Original image", None),
    (overlay_tiled, f"Sea-ice floes only\n{len(masks_tiled_floes)} masks", None),
    (labels_tiled, "Floe-label mask", "nipy_spectral"),
]
for ax, (img, title, cmap) in zip(axes[:3], panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

if len(diameters_tiled_m) >= 2:
    bin_edges = np.logspace(np.log10(diameters_tiled_m.min()),
                            np.log10(diameters_tiled_m.max()), 30)
    pdf_tiled, edges = np.histogram(diameters_tiled_m, bins=bin_edges, density=True)
    centers = np.sqrt(edges[:-1] * edges[1:])
    valid = pdf_tiled > 0
    axes[3].plot(centers[valid], pdf_tiled[valid], marker="o")
    axes[3].set(xscale="log", yscale="log",
                xlabel="Equivalent floe diameter $D_f$ [m]",
                ylabel="Probability density $p(D_f)$ [1/m]",
                title="Floe-size distribution")
    axes[3].grid(True, which="both", alpha=0.3)
else:
    axes[3].text(0.5, 0.5, "Not enough floes\nfor FSD",
                 ha="center", va="center", transform=axes[3].transAxes)
    axes[3].set_title("Floe-size distribution")

plt.suptitle(f"Trial 2 — {N}×{N} tiled SAM | image width = {IMAGE_WIDTH_M:.0f} m", fontsize=16)
plt.tight_layout()
plt.show()

if len(diameters_tiled_m):
    print("\nFLOE-SIZE STATISTICS — TRIAL 2\n==============================")
    print(f"Minimum diameter: {diameters_tiled_m.min():.2f} m")
    print(f"Median diameter:  {np.median(diameters_tiled_m):.2f} m")
    print(f"Mean diameter:    {np.mean(diameters_tiled_m):.2f} m")
    print(f"Maximum diameter: {diameters_tiled_m.max():.2f} m")


# Compare Trial 1 and Trial 2

- **Trial 1:** SAM processes the full image once.
- **Trial 2:** SAM processes \(N\times N\) tiles independently.

More masks do not necessarily mean better segmentation; they can represent real small floes, over-segmentation, or tile-boundary artifacts.


## 16. Side-by-side visual comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
for ax, img, title in [
    (axes[0], image, "Original image"),
    (axes[1], overlay_full, f"Trial 1 — Full image\n{len(masks_full)} masks"),
    (axes[2], overlay_tiled, f"Trial 2 — {N}×{N} tiles\n{len(masks_tiled)} masks"),
]:
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 17. Compare mask counts, mask areas, and runtime


In [ ]:
areas_full = np.array([m["area"] for m in masks_full])
areas_tiled = np.array([m["area"] for m in masks_tiled])

for title, masks, areas, runtime in [
    ("TRIAL 1 — FULL IMAGE", masks_full, areas_full, time_full),
    (f"TRIAL 2 — {N}×{N} TILES", masks_tiled, areas_tiled, time_tiled),
]:
    print(title)
    print("-" * len(title))
    print("Number of masks:", len(masks))
    print(f"Runtime: {runtime:.1f} s")
    if len(areas):
        print(f"Median mask area: {np.median(areas):.1f} pixels")
        print(f"Smallest mask:    {areas.min():.0f} pixels")
        print(f"Largest mask:     {areas.max():.0f} pixels")
    print()

if masks_full:
    change = 100 * (len(masks_tiled) - len(masks_full)) / len(masks_full)
    print(f"Change in number of masks: {change:+.1f}%")


## 18. Direct comparison of physical floe-size distributions


In [ ]:
# Compare the cleaned physical floe diameters from Trials 1 and 2.
D_full = np.asarray(diameters_full_m, dtype=float)
D_tiled = np.asarray(diameters_tiled_m, dtype=float)
D_full = D_full[np.isfinite(D_full) & (D_full > 0)]
D_tiled = D_tiled[np.isfinite(D_tiled) & (D_tiled > 0)]
if len(D_full) == 0 or len(D_tiled) == 0:
    raise RuntimeError("Both trials must contain at least one valid floe before comparing their FSDs.")

print("DIRECT FSD COMPARISON\n=====================")
for title, D in [("TRIAL 1 — FULL IMAGE", D_full), (f"TRIAL 2 — {N}×{N} TILED", D_tiled)]:
    print(f"\n{title}\n{'-' * len(title)}")
    print(f"Number of floes: {len(D)}")
    print(f"Minimum D_f:     {D.min():.2f} m")
    print(f"Median D_f:      {np.median(D):.2f} m")
    print(f"Mean D_f:        {np.mean(D):.2f} m")
    print(f"Maximum D_f:     {D.max():.2f} m")

# Common logarithmic bins for direct PDF comparison.
all_D = np.concatenate([D_full, D_tiled])
D_min, D_max = all_D.min(), all_D.max()
N_BINS = 35
if np.isclose(D_min, D_max):
    D_min, D_max = 0.9 * D_min, 1.1 * D_max

bin_edges = np.logspace(np.log10(D_min), np.log10(D_max), N_BINS + 1)
bin_widths = np.diff(bin_edges)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])
counts_full, _ = np.histogram(D_full, bins=bin_edges)
counts_tiled, _ = np.histogram(D_tiled, bins=bin_edges)
pdf_full = counts_full / (len(D_full) * bin_widths)
pdf_tiled = counts_tiled / (len(D_tiled) * bin_widths)

plt.figure(figsize=(9, 7))
for D, pdf, label in [
    (D_full, pdf_full, f"Trial 1: full image (n={len(D_full)})"),
    (D_tiled, pdf_tiled, f"Trial 2: {N}×{N} tiled (n={len(D_tiled)})"),
]:
    valid = pdf > 0
    plt.plot(bin_centers[valid], pdf[valid], marker="o",
             linewidth=2, markersize=5, label=label)

plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"Equivalent floe diameter, $D_f$ [m]", fontsize=12)
plt.ylabel(r"Probability density, $p(D_f)$ [m$^{-1}$]", fontsize=12)
plt.title("Floe-Size Distribution: Full Image vs Tiled SAM")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

integral_full = np.sum(pdf_full * bin_widths)
integral_tiled = np.sum(pdf_tiled * bin_widths)
median_change_percent = (np.median(D_tiled) - np.median(D_full)) / np.median(D_full) * 100

print("\nPDF NORMALIZATION\n=================")
print(f"Trial 1 integral: {integral_full:.6f}")
print(f"Trial 2 integral: {integral_tiled:.6f}")
print("\nTRIAL 2 RELATIVE TO TRIAL 1\n===========================")
print(f"Additional detected/retained floes: {len(D_tiled) - len(D_full):+d}")
print(f"Ratio of retained floe counts: {len(D_tiled) / len(D_full):.2f}")
print(f"Change in median floe diameter: {median_change_percent:+.1f}%")


## 19. Zoom into a region to inspect small floes

Compare the same zoomed region in the original image and both segmentation trials.


In [ ]:
# Fractional coordinates of the zoom region; edit to inspect another area.
x_fraction_min, x_fraction_max = 0.25, 0.55
y_fraction_min, y_fraction_max = 0.25, 0.55
zx0, zx1 = int(W * x_fraction_min), int(W * x_fraction_max)
zy0, zy1 = int(H * y_fraction_min), int(H * y_fraction_max)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, img, title in [
    (axes[0], image, "Original — zoomed"),
    (axes[1], overlay_full, "Trial 1 — full-image SAM"),
    (axes[2], overlay_tiled, f"Trial 2 — {N}×{N} tiled SAM"),
]:
    ax.imshow(img[zy0:zy1, zx0:zx1])
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 20. Inspect possible tile-boundary artifacts

Overlay the tile boundaries on Trial 2 to inspect floes that may have been split across neighboring tiles.


In [ ]:
plt.figure(figsize=(11, 11))
plt.imshow(overlay_tiled)
for x in x_edges[1:-1]:
    plt.axvline(x, linewidth=2)
for y in y_edges[1:-1]:
    plt.axhline(y, linewidth=2)
plt.title(f"Trial 2 segmentation with {N}×{N} tile boundaries")
plt.axis("off")
plt.show()


## 21. Save both segmentation results


In [ ]:
OUTPUT_DIR = "/content/SAM_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FULL_OVERLAY_PATH = os.path.join(OUTPUT_DIR, "trial1_full_image_overlay.png")
FULL_LABEL_PATH = os.path.join(OUTPUT_DIR, "trial1_full_image_labels.png")
TILED_OVERLAY_PATH = os.path.join(OUTPUT_DIR, f"trial2_tiled_N{N}_overlay.png")
TILED_LABEL_PATH = os.path.join(OUTPUT_DIR, f"trial2_tiled_N{N}_labels.png")

cv2.imwrite(FULL_OVERLAY_PATH, cv2.cvtColor(overlay_full, cv2.COLOR_RGB2BGR))
cv2.imwrite(TILED_OVERLAY_PATH, cv2.cvtColor(overlay_tiled, cv2.COLOR_RGB2BGR))
cv2.imwrite(FULL_LABEL_PATH, labels_full)
cv2.imwrite(TILED_LABEL_PATH, labels_tiled)

print("Saved results:")
for path in [FULL_OVERLAY_PATH, FULL_LABEL_PATH, TILED_OVERLAY_PATH, TILED_LABEL_PATH]:
    print(path)


# References & Further Reading:
